In [2]:
import polars as pl
from pathlib import Path
import shutil
import dotenv
from Bio import Entrez

In [3]:
dotenv.load_dotenv()

email = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_EMAIL")
if email is None:
    raise ValueError("ENTREZ_EMAIL not found in .env file or environment variables.")
Entrez.email = email
print(f"Entrez email set to: {email}")

api_key = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_API_KEY")
if api_key is None:
    print("ENTREZ_API_KEY not found in .env file, proceeding without it.")
else:
    print("Entrez API key set.")
    Entrez.api_key = api_key

Entrez email set to: dupin@alunos.utfpr.edu.br
Entrez API key set.


In [4]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

Folder where groups (data) folder should be: /data/users/bdupin/sam-honours


In [5]:
source_dir = Path("../igannot/sandbox/new_atlas").resolve()
source_dir

PosixPath('/data/users/bdupin/igannot/sandbox/new_atlas')

In [6]:
groups_map = {
    "plants_plt_solanaceae": "plants_plt"
}

In [ ]:
for g_dest, g_src in groups_map.items():
    dest_dir = BASE / g_dest
    source_dir = source_dir / g_src

    tsv_src = source_dir / f"{g_src}.tsv"
    tsv_dest = dest_dir / f"{g_dest}.tsv"

    shutil.copy(tsv_src, tsv_dest)


In [ ]:
for g in groups_map.keys():
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")

    taxids = tsv["ncbi_taxid"].unique().to_list()

In [ ]:
len(taxids)

In [ ]:
query_size = 250

taxid_map = {}

for window in range(0, len(taxids), query_size):
    taxid_window = taxids[window:window + query_size]
    ids = ",".join(map(str, taxid_window))
    print(f"Processing taxids {window} to {window + query_size} (total {len(taxid_window)})")

    with Entrez.efetch(db="taxonomy", id=ids) as handle:
        records = Entrez.read(handle)

    for record in records:
        print(f"TaxId: {record['TaxId']}, Lineage: {record['Lineage']}")
        taxid_map[record["TaxId"]] = record["Lineage"]

In [ ]:
tsv = tsv.with_columns(
    pl.col("ncbi_taxid").replace_strict(taxid_map, default="Unknown").alias("lineage")
)

## filtering Lineage

In [10]:
keywors = ["Solanales", "Solanaceae"]

for g, gsoure in groups_map.items():
    tsv = pl.read_csv(BASE / g / f"{gsoure}_raw_lineage.tsv", separator="\t")
    tsv = tsv.filter(pl.col("lineage").str.contains("|".join(keywors))).drop("lineage")

    # remove duplicates based on ncbi_taxid, keeping largest Genome_length value
    # count how many duplicates were removed
    num_duplicates = tsv.shape[0] - tsv.unique(subset=["ncbi_taxid"], keep="first").shape[0]
    print(f"Removed {num_duplicates} duplicates based on ncbi_taxid for group {g}")
    tsv = tsv.sort("Genome_length", descending=True).unique(subset=["ncbi_taxid"], keep="first")
    tsv = tsv.sort("AN")

    tsv.write_csv(BASE / g / f"{g}.tsv", separator="\t")

Removed 37 duplicates based on ncbi_taxid for group plants_plt_solanaceae


## getting atlas

In [ ]:
for g_dest, g_src in groups_map.items():

    atlas_src = source_dir / g_src / f"{g_src}_summary_atlas.tsv"
    atlas_dest = BASE / g_dest / f"{g_src}_summary_atlas.tsv"

    shutil.copy(atlas_src, atlas_dest)

In [11]:
for g, g_src in groups_map.items():
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")

    atlas = pl.read_csv(BASE / g / f"{g_src}_summary_atlas.tsv", separator="\t")

    # filter atlas with AN only present in tsv
    atlas = atlas.filter(pl.col("AN").is_in(tsv["AN"].implode()))

    atlas = atlas.with_columns(
        pl.col("UP").str.replace("PLT-", "").alias("UP"),
        pl.col("DOWN").str.replace("PLT-", "").alias("DOWN"),
    )

    atlas = atlas.with_columns(
        pl.concat_str([pl.col("UP"), pl.col("DOWN"), pl.col("Polarity")], separator="|").alias("gene_pair")
    )
    atlas.write_csv(BASE / g / f"{g}_summary_atlas.tsv", separator="\t")